# Build a Tiny Transformer From Scratch (PyTorch)

This notebook is designed for **momentum**: you’ll *build and run code quickly*, but still understand what each piece does.

You will:
1. Warm up with a tiny **MLP** (so training loop + backprop feels concrete)
2. Implement a simple **character-level tokenizer**
3. Build a minimal **GPT-style Transformer** (causal self-attention + FFN)
4. Train on a small text corpus and **generate** text

> Works on CPU. If you have a GPU, it will use it automatically.


In [3]:
import math, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cpu'

## Part 1 — Warm-up: a tiny MLP (feel the training loop)

This section is just to make sure the forward → loss → backward → update loop is working and intuitive.
We’ll solve a tiny XOR-like classification problem.


In [2]:
# synthetic dataset: XOR-ish (not linearly separable)
X = torch.tensor([[0.,0.],[0.,1.],[1.,0.],[1.,1.]], device=device)
y = torch.tensor([0,1,1,0], device=device)  # classes 0/1

mlp = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 2),
).to(device)

opt = torch.optim.Adam(mlp.parameters(), lr=0.05)

for step in range(300):
    logits = mlp(X)
    loss = F.cross_entropy(logits, y)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    if step % 50 == 0:
        pred = logits.argmax(dim=1)
        acc = (pred == y).float().mean().item()
        print(f"step={step:3d} loss={loss.item():.4f} acc={acc:.2f}")


step=  0 loss=0.6957 acc=0.50
step= 50 loss=0.0013 acc=1.00
step=100 loss=0.0005 acc=1.00
step=150 loss=0.0003 acc=1.00
step=200 loss=0.0002 acc=1.00
step=250 loss=0.0002 acc=1.00


## Part 2 — Tokenizer (character-level)

To move fast, we’ll start with a **character-level** tokenizer:
- Vocabulary = set of characters appearing in the training text
- `encode(text)` → tensor of token IDs
- `decode(ids)` → back to text

This is the simplest tokenizer and perfect for a first transformer build.


In [ ]:
text = """In the beginning there was code.
And the code became a model.
And the model began to talk.
""" * 80  # repeat to have enough training data

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

def encode(s: str) -> torch.Tensor:
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

def decode(ix: torch.Tensor) -> str:
    return "".join([itos[i.item()] for i in ix])

data = encode(text).to(device)

vocab_size, len(data), chars[:12]


## Part 3 — A minimal GPT-style Transformer (from scratch)

A GPT-like block (decoder-only transformer) contains:
1. **Causal self-attention** (each position attends only to itself and the past)
2. **Feed-forward network (FFN)** — a classic MLP applied to each position
3. Residual connections + LayerNorm

We’ll stack a few blocks and train it as a next-character language model.


### 3.1 Causal self-attention (the core new ingredient)

Given input `x` of shape `(B, T, C)`:
- compute queries/keys/values: `Q, K, V`
- attention weights: `softmax(QK^T / sqrt(d))`
- apply **causal mask** so a token cannot look ahead
- produce output as weighted sum of values

Multi-head attention = do this in parallel in multiple subspaces, then concatenate.


In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float, block_size: int):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        # causal mask (1s in allowed positions)
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.split(C, dim=2)

        # reshape to heads: (B, nh, T, hd)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # attention scores (B, nh, T, T)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # apply causal mask: set future positions to -inf
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))

        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v  # (B, nh, T, hd)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # back to (B, T, C)

        y = self.resid_drop(self.proj(y))
        return y


### 3.2 Feed-forward network (FFN)

This is the “classical neural net” inside the transformer:
\[
\text{FFN}(x) = W_2\,\sigma(W_1 x + b_1) + b_2
\]

We typically expand dimension (e.g., 4×) then compress back.


In [ ]:
class FFN(nn.Module):
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


### 3.3 Transformer block (Attention + FFN)

We use **Pre-LN** (LayerNorm before each sublayer) which is stable and common in modern LLMs.


In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float, block_size: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FFN(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


### 3.4 Full language model

Components:
- token embeddings
- positional embeddings
- a stack of blocks
- final LayerNorm
- output projection to vocab size

Training objective:
- next-token prediction using cross-entropy


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size: int, block_size: int, n_embd: int, n_head: int, n_layer: int, dropout: float):
        super().__init__()
        self.block_size = block_size

        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            Block(n_embd, n_head, dropout, block_size) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        # idx: (B, T)
        B, T = idx.shape
        assert T <= self.block_size, "Sequence length exceeds block size."

        pos = torch.arange(0, T, device=idx.device)  # (T,)
        x = self.tok_emb(idx) + self.pos_emb(pos)    # (B, T, C)
        x = self.drop(x)

        for blk in self.blocks:
            x = blk(x)

        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab)

        loss = None
        if targets is not None:
            # flatten batch+time
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0, top_k: int | None = None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]  # crop to block size
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature  # last time step

            if top_k is not None:
                v, _ = torch.topk(logits, k=top_k, dim=-1)
                logits = logits.masked_fill(logits < v[:, [-1]], float("-inf"))

            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx


## Part 4 — Data batching for next-token prediction

We’ll sample random chunks of length `block_size` from our tokenized data.
Inputs `x` predict targets `y` which are the same sequence shifted by 1.


In [ ]:
block_size = 64   # context length
batch_size = 64

def get_batch(data_1d: torch.Tensor):
    # sample starting indices
    ix = torch.randint(0, len(data_1d) - block_size - 1, (batch_size,), device=device)
    x = torch.stack([data_1d[i:i+block_size] for i in ix])
    y = torch.stack([data_1d[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch(data)
xb.shape, yb.shape


## Part 5 — Train the tiny transformer

We’ll keep it small so it trains quickly:
- embedding size: `n_embd`
- heads: `n_head`
- layers: `n_layer`

Then we’ll periodically generate text.


In [ ]:
# model hyperparameters (small on purpose)
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1

model = TinyGPT(vocab_size, block_size, n_embd, n_head, n_layer, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

sum(p.numel() for p in model.parameters())/1e6


In [ ]:
def estimate_loss(model, data_1d, iters=50):
    model.eval()
    losses = []
    with torch.no_grad():
        for _ in range(iters):
            x, y = get_batch(data_1d)
            _, loss = model(x, y)
            losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

# quick sanity check: forward pass
x, y = get_batch(data)
logits, loss = model(x, y)
logits.shape, loss.item()


In [ ]:
max_steps = 800
eval_every = 200

t0 = time.time()
for step in range(1, max_steps+1):
    x, y = get_batch(data)
    _, loss = model(x, y)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # mild stability
    optimizer.step()

    if step % eval_every == 0 or step == 1:
        avg = estimate_loss(model, data, iters=30)
        dt = time.time() - t0
        print(f"step={step:4d} train_loss={loss.item():.3f} est_loss={avg:.3f} time={dt:.1f}s")

        # generate a sample
        prompt = "In the "
        idx = encode(prompt).unsqueeze(0).to(device)
        out = model.generate(idx, max_new_tokens=120, temperature=1.0, top_k=20)[0]
        print("\n--- sample ---")
        print(decode(out))
        print("--------------\n")


## Next steps (optional upgrades)

If you want to keep momentum but get closer to “real” training patterns:
- Add a **train/val** split
- Increase data size (load a `.txt` file)
- Add **learning rate schedule** (warmup + cosine decay)
- Try **byte-level** tokenization (still simple)
- Implement **manual attention** checks (print attention weights for a small batch)
- Move to a **BPE tokenizer** later (e.g., tiktoken) once the core model feels natural
